**Cosine encoder → HistGB optimization.** Architecture from the saved winner in `opt.ipynb` (width 32, five layers, eight heads), training parameters from the confirmed winner in `cosine_annealing_opt.ipynb` (trial 23), and the starting HistGB parameters from `histgb_opt.ipynb`.

Use five disjoint patient folds and seed 42. Train each encoder once on all fold-training patients for **160 fixed epochs**, with 40-epoch cosine cycles, T_mult=1 and a minimum learning rate of 1% of the initial rate. No internal encoder holdout or epoch selection. Fit preprocessing and scaling only on fold-training patients.

Cache the engineered features plus frozen embeddings, then run **40 Optuna trials** tuning HistGB only. The first trial uses its previous best parameters. These are tuning scores on the same folds used for selection, rather than an independent test estimate.


In [1]:
import ast
import json
from pathlib import Path

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
encoder_definitions = json.loads(
    (PROJECT_DIR / "exploration/stacking/cosine_annealing.ipynb").read_text()
)
# Reuse setup and encoder functions only; no earlier validation is run.
for i in [2, 4, 6, 8, 10]:
    exec("".join(encoder_definitions["cells"][i]["source"]))

FIXED_EPOCHS, N_TRIALS = 160, 40
# Architecture from opt; training parameters from cosine_annealing_opt (trial 23).
ENCODER = dict(width=32, heads=8, layers=5, dropout=0.19181651819990855,
               ff_multiplier=3, mlp_layers=2, mlp_units=16)
LEARNING_RATE, WEIGHT_DECAY = 0.0046066691407564055, 0.0026062405166073018
RESTART_PERIOD, RESTART_MULTIPLIER = 40, 1
MIN_LEARNING_RATE = LEARNING_RATE * 0.01
# Starting point from histgb_opt.
HISTGB_PARAMS = dict(
    learning_rate=0.03471523676107423, max_iter=800, max_leaf_nodes=62,
    min_samples_leaf=16, l2_regularization=8.305016398991823,
    max_features=0.6408427464001498,
)
baseline[-1].set_params(random_state=SEED, **HISTGB_PARAMS)

# Import only the fixed-duration helper, without its comparison/training loop.
fixed_source = "".join(next(c for c in encoder_definitions["cells"]
                            if c.get("id") == "cosine-fixed-cv")["source"])
fixed_function = next(f for f in ast.parse(fixed_source).body
                      if isinstance(f, ast.FunctionDef) and f.name == "fit_encoder_full")
exec(ast.get_source_segment(fixed_source, fixed_function))


Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=8.305016398991823,
                              learning_rate=0.03471523676107423,
                              max_features=0.6408427464001498, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=16,
                              random_state=42)


In [ ]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tuning_folds = []
with threadpool_limits(limits=3):
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y, groups=X["patient_id"]), 1):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        train, validation = prep.fit_transform(raw_train, yt), prep.transform(raw_val)
        fitted_encoder = fit_encoder_full(raw_train, yt, prep[:2], SEED, FIXED_EPOCHS)
        z_train, _ = encoder_outputs(fitted_encoder, raw_train)
        z_val, _ = encoder_outputs(fitted_encoder, raw_val)
        tuning_folds.append((train.join(z_train), validation.join(z_val), yt, yv))
        print(f"Fold {fold}/5: encoder trained for {FIXED_EPOCHS} epochs; embeddings cached.")


In [ ]:
import optuna
from threadpoolctl import threadpool_limits

optuna.logging.set_verbosity(optuna.logging.INFO)

# Reuse the fixed encoder embeddings from five patient folds and one seed.
def histgb_objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "max_iter": trial.suggest_int("max_iter", 100, 800, step=100),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 7, 63, log=True),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 120, log=True),
        "l2_regularization": trial.suggest_float("l2_regularization", 0, 30),
        "max_features": trial.suggest_float("max_features", 0.6, 1.0),
    }
    scores = []
    for step, (train, validation, yt, yv) in enumerate(tuning_folds):
        fitted = clone(baseline[-1]).set_params(**params).fit(train, yt)
        scores.append(mean_squared_error(yv, fitted.predict(validation)))
        # Report each patient fold; prune after at least two folds.
        if (step + 1) % len(SEEDS) == 0:
            trial.report(float(np.mean(scores)), step // len(SEEDS))
            if step < len(tuning_folds) - 1 and trial.should_prune():
                raise optuna.TrialPruned()
    trial.set_user_attr("fold_mse", scores)
    return float(np.mean(scores))


histgb_study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1),
)
parameter_names = ["learning_rate", "max_iter", "max_leaf_nodes", "min_samples_leaf", "l2_regularization", "max_features"]
histgb_study.enqueue_trial({name: baseline[-1].get_params()[name] for name in parameter_names})
with threadpool_limits(limits=3):
    histgb_study.optimize(histgb_objective, n_trials=N_TRIALS)

baseline_scores = histgb_study.trials[0].user_attrs["fold_mse"]
best_scores = histgb_study.best_trial.user_attrs["fold_mse"]
tuning_summary = pd.DataFrame([
    {"model": name, "MSE": np.mean(scores), "std": np.std(scores)}
    for name, scores in [("Encoder_HistGB_current", baseline_scores), ("Encoder_HistGB_tuned", best_scores)]
]).set_index("model")
tuning_summary["RMSE"] = np.sqrt(tuning_summary["MSE"])
tuning_summary["gain_MSE_pct"] = 100 * (1 - tuning_summary["MSE"] / np.mean(baseline_scores))
best_histgb = clone(baseline[-1]).set_params(**histgb_study.best_params)
print("Best HistGB parameters:", histgb_study.best_params)
tuning_summary
